In [ ]:
import json
import os
from pathlib import Path
from typing import Any

import pandas as pd

from kibad_llm.config import PROJ_ROOT

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)
# set wider column width for displaying pandas data frames
pd.set_option("max_colwidth", 400)

In [ ]:
EVALUATE_DIR = "data/prediction_results/159_core_schema_baseline_gpt5/logs/evaluate"

In [ ]:
# load evaluation job_return_value.json files


def _load_evaluations(parent_dir: Path) -> dict:
    log_filename = "job_return_value.json"
    # get sub directories, 1 level only
    run_dirs = [p for p in Path(parent_dir).iterdir() if p.is_dir()]
    # assume that each subdir contains a 'job_return_value.json' from a multi-run evaluation
    # (i.e. it contains a parent key indictating the run, may be empty, and sub-keys 'prediction' and 'overrides' that store metadata)
    data = [json.loads((subdir / log_filename).read_text()) for subdir in run_dirs]
    # keep the keys / identifiers?
    # runs = {key: subdict for d in data for key, subdict in d.items()}
    runs = [subdict for d in data for subdict in d.values()]
    return runs


eval_runs = _load_evaluations(EVALUATE_DIR)

In [ ]:
# simplify the dictionaries and add group keys


def _flatten_dict(d: dict[str, Any], sep: str = ".") -> dict[str, Any]:
    result: dict[str, Any] = dict()
    for k, v in d.items():
        if isinstance(v, dict):
            for e in v:
                result[f"{k}{sep}{e}"] = v[e]
    return result


def _clean_metadata(run_dict: dict) -> (dict, dict):
    metadata_keys = ["prediction", "overrides"]
    metadata = {}
    for k in metadata_keys:
        metadata[k] = run_dict.pop(k)

    return _flatten_dict(run_dict), _flatten_dict(metadata)


metrics, metadata = map(list, zip(*[_clean_metadata(run) for run in eval_runs]))

In [ ]:
metrics_df = pd.DataFrame.from_records(metrics)

In [ ]:
# add some metadata keys
metrics_df["overrides.pdf_directory"] = [m["overrides.pdf_directory"] for m in metadata]
metrics_df["overrides.extractor/llm"] = [m["overrides.extractor/llm"] for m in metadata]
metrics_df["overrides.experiment/predict"] = [m["overrides.experiment/predict"] for m in metadata]
metrics_df

In [ ]:
cols_to_plot = [
    col for col in metrics_df.columns if col.endswith("f1") and not col.startswith("AVG")
]
metrics_df.plot(kind="bar", x="overrides.extractor/llm", y=cols_to_plot)